# 1725952349_ent - Question and Answer Dataset

Generated starter notebook from [DesiData](https://www.desidata.in) — India's open dataset catalogue. Run the cells top to bottom.

177 extractive question-and-answer pairs built from 1725952349 ent, published by icmr.gov.in. The source is a 17-page publication. Source-evidenced topics include standard treatment workflows, acute rhinosinusitis, chronic rhinosinusitis, epistaxis, universal health coverage, pmjay, ayushman bharat, icd 10 j01.90. Every answer is a verbatim span of text the source prints, and each row carries the passage it sits in, its offset in that passage, the source quote, the page and the location in the document, so any row can be checked against the original. Question types in the export: 96 factual, 43 list, 16 definition, 11 summary, 9 relationship, 2 comparison. 81 of the 177 pairs (45.8%) ask explanatory questions; the remaining 96 are factual or table-based questions. 100.00% of rows pass the corpus quality gate.

Source: icmr.gov.in  
Licence: Not specified  
Dataset page: https://www.desidata.in/datasets/www-icmr-gov-in-501f003dceb9877d3b13b8695c5d9770

In [ ]:
!pip install pandas matplotlib --quiet

## Load the dataset

Dataset downloads require a free DD token. Create one in Profile & settings, then set `DD_TOKEN` in your environment.
In Colab, add `DD_TOKEN` under Secrets and enable notebook access to that secret.

In [ ]:
import io
import json
import os
import pandas as pd
import urllib.request

url = "https://www.desidata.in/api/datasets/www-icmr-gov-in-501f003dceb9877d3b13b8695c5d9770/download"
if "COLAB_RELEASE_TAG" in os.environ and "DD_TOKEN" not in os.environ:
    from google.colab import userdata
    os.environ["DD_TOKEN"] = userdata.get("DD_TOKEN")
request = urllib.request.Request(url, headers={"Authorization": f"Bearer {os.environ['DD_TOKEN']}", "Accept": "application/json"})
with urllib.request.urlopen(request) as response:
    file_url = json.load(response)["download_url"]
# Fetch the signed file URL separately; never send DD_TOKEN to the storage host.
with urllib.request.urlopen(file_url) as response:
    df = pd.read_csv(io.BytesIO(response.read()))

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## Explore

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## See one example

Every row pairs a question with the text it was written from.

In [ ]:
row = df.iloc[0]

print("QUESTION:", row["question"])

print("ANSWER:", str(row["answer"])[:400])

print()
print("SOURCE CONTEXT:")
print(str(row["context"])[:600])

## Mini baseline — can TF-IDF find the source paragraph?

Each question was written from its row's source paragraph. A simple TF-IDF
retriever has to rank the correct one first. Its hit rate is the number
any trained model on this dataset must beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

sample = df.head(300)
questions = sample["question"].astype(str).to_numpy()
passages = sample["context"].astype(str).to_numpy()

vectorizer = TfidfVectorizer(stop_words="english")
passage_vectors = vectorizer.fit_transform(passages)
question_vectors = vectorizer.transform(questions)

best = np.empty(len(questions), dtype=int)
for i in range(len(questions)):
    best[i] = linear_kernel(question_vectors[i], passage_vectors).argmax()

hit_rate = (passages[best] == passages).mean()
print(f"TF-IDF retrieval lands on the correct paragraph for {hit_rate:.0%} of questions")
print("That is the number a trained model has to beat.")

## What will you build?

- Look for gaps across regions, age groups or demographics
- Build a simple risk score and validate it honestly on a hold-out set
- Chart how an indicator changes over time if the data has a year column

- Browse more [Health datasets](https://www.desidata.in/datasets?category=Health) or the [full catalogue](https://www.desidata.in/datasets).
- Sharing your work? Tag DesiData — we feature community notebooks built on this catalogue.

In [ ]:
# Data via DesiData — India's open dataset catalogue
print("Dataset page:", "https://www.desidata.in/datasets/www-icmr-gov-in-501f003dceb9877d3b13b8695c5d9770")